# Notebook 01 - Transcript Ingestion Pipeline

**Project:** IncidentIQ - AI-powered Incident Intelligence

**Goal:** Transform a YouTube video into a searchable knowledge base stored in Pinecone cloud.

## What this notebook covers
1. Install and import all required packages
2. Configuration and Pinecone initialization
3. Fetch transcript from YouTube
4. Clean and chunk the transcript
5. Store chunks in Pinecone cloud
6. Verify with a first similarity search
7. Run quality tests

## Why Pinecone instead of ChromaDB
- Cloud-based — no local storage needed
- Accessible from any machine — perfect for team collaboration and deployment
- Data persists permanently — no re-fetching from YouTube
- Each video is fetched exactly once and cached forever

---

## 1. Install required packages

Installing all dependencies including Pinecone cloud vector database.
Run this cell once — skip on subsequent runs.

In [ ]:
!pip install youtube-transcript-api langchain langchain-openai langchain-community langchain-text-splitters langchain-pinecone pinecone python-dotenv -q
print('Packages installed.')

## 2. Import libraries and load environment variables

We load API keys from a .env file.
Required keys: OPENAI_API_KEY and PINECONE_API_KEY.

In [ ]:
import os
import re
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not found. Check your .env file.'
assert os.getenv('PINECONE_API_KEY'), 'PINECONE_API_KEY not found. Check your .env file.'
print('Environment variables loaded successfully.')

## 3. Configuration

Central configuration for the pipeline.
Pinecone index name must match the index created in the Pinecone dashboard.
Chunk size and overlap control how the transcript is split for retrieval.

In [ ]:
# Video to process
YOUTUBE_URL     = 'https://www.youtube.com/watch?v=7OH5FEWWM_k'

# Pinecone settings — must match your Pinecone dashboard
INDEX_NAME      = 'incidentiq'
EMBEDDING_MODEL = 'text-embedding-3-small'  # 1536 dimensions — matches Pinecone index

# Chunking settings
CHUNK_SIZE      = 500   # Max characters per chunk
CHUNK_OVERLAP   = 50    # Overlap preserves context at chunk boundaries

def extract_video_id(url: str) -> str:
    'Extract YouTube video ID from standard or shortened URL.'
    if 'v=' in url:
        return url.split('v=')[1].split('&')[0]
    elif 'youtu.be/' in url:
        return url.split('youtu.be/')[1].split('?')[0]
    raise ValueError(f'Cannot extract video ID from URL: {url}')

video_id = extract_video_id(YOUTUBE_URL)

# Initialize embedding model
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

# Initialize vectorstore
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    pinecone_api_key=os.getenv('PINECONE_API_KEY'),
)

print('Configuration set.')
print(f'   Video ID:        {video_id}')
print(f'   Pinecone index:  {INDEX_NAME}')
print(f'   Embedding model: {EMBEDDING_MODEL}')

## 4. Check Pinecone cache — fetch from YouTube only if needed

This cell checks if the video is already stored in Pinecone before making any YouTube request.
Each video is fetched from YouTube exactly once and stored permanently in Pinecone.
Subsequent runs load from Pinecone cache — no YouTube call needed.

This prevents IP blocking issues caused by repeated YouTube requests.

In [ ]:
def clean_transcript(text: str) -> str:
    'Remove noise tags and empty timestamps from YouTube transcripts.'
    text = re.sub(r'\[Music\]|\[Applause\]|\[Laughter\]|\[Cheering\]', '', text)
    text = re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def fetch_from_youtube(video_id: str) -> tuple:
    'Fetch and clean transcript from YouTube. Called only if not in Pinecone cache.'
    try:
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id, languages=['en', 'nl', 'fr'])
        transcript_list = fetched.snippets
    except NoTranscriptFound:
        raise ValueError(
            f'No transcript found for video {video_id}.\n'
            'Please use a video with subtitles enabled (CC button visible in YouTube).'
        )
    except TranscriptsDisabled:
        raise ValueError(f'Transcripts are disabled for video {video_id}.')
    except Exception as e:
        raise ValueError(
            f'Could not fetch transcript.\n'
            f'If YouTube is blocking your IP, wait 30-60 minutes and try again.\n'
            f'Error: {str(e)}'
        )

    plain = ' '.join(t.text for t in transcript_list)
    timestamped = ' '.join(
        f'[{int(t.start//60):02d}:{int(t.start%60):02d}] {t.text}'
        for t in transcript_list
    )
    return clean_transcript(plain), clean_transcript(timestamped)


# Check Pinecone cache first
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
existing_count = stats.total_vector_count

if existing_count > 0:
    print(f'Video already in Pinecone — {existing_count} vectors found.')
    print('Skipping YouTube fetch — using cached data.')
    print('Ready for Q&A.')
    plain_text = 'Loaded from Pinecone cache'
    timestamped_text = plain_text
else:
    print('Video not in Pinecone — fetching from YouTube...')
    plain_text, timestamped_text = fetch_from_youtube(video_id)
    print(f'Transcript fetched successfully.')
    print(f'   Total characters: {len(plain_text):,}')
    print(f'\nPreview (first 300 chars):')
    print('-' * 60)
    print(plain_text[:300])

## 5. Split transcript into chunks

We split the transcript into overlapping chunks for accurate retrieval.
This cell is skipped if the video was already in Pinecone cache.

Why RecursiveCharacterTextSplitter:
Splits on natural boundaries first (paragraphs, sentences, words)
before splitting mid-word — preserving semantic coherence in each chunk.

In [ ]:
if existing_count > 0:
    print('Skipping chunking — video already in Pinecone.')
else:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '. ', ' '],
    )
    chunks = splitter.create_documents(
        texts=[timestamped_text],
        metadatas=[{'video_id': video_id, 'source': YOUTUBE_URL}]
    )
    print(f'Transcript split into {len(chunks)} chunks.')
    print(f'   Chunk size:    {CHUNK_SIZE} characters')
    print(f'   Chunk overlap: {CHUNK_OVERLAP} characters')
    print(f'\nExample chunk 3:')
    print('-' * 60)
    print(chunks[2].page_content)
    print(f'\nMetadata: {chunks[2].metadata}')

## 6. Store chunks in Pinecone

Each chunk is embedded using the OpenAI model and stored in Pinecone.
This process runs only once per video — subsequent runs use the cache.

Pinecone automatically handles duplicate prevention via upsert.
The video_id in metadata allows filtering queries to a specific video.

In [ ]:
if existing_count > 0:
    print('Skipping storage — video already in Pinecone.')
else:
    print(f'Embedding and storing {len(chunks)} chunks in Pinecone...')
    print('This may take 20-30 seconds...')
    vectorstore.add_documents(chunks)
    print('All chunks stored successfully in Pinecone.')
    print(f'   Index:  {INDEX_NAME}')
    print(f'   Chunks: {len(chunks)}')

## 7. Verify — first similarity search

We run a semantic search to confirm the pipeline works end-to-end.
The query is embedded and compared against all stored vectors using cosine similarity.
This is the exact retrieval mechanism the RAG agent uses in production.

In [ ]:
TEST_QUERY = 'What mistakes were made and what are the key lessons learned?'
results = vectorstore.similarity_search(TEST_QUERY, k=3)

print(f'Query: {TEST_QUERY}')
print(f'\nTop {len(results)} most relevant chunks:')
print('=' * 60)
for i, doc in enumerate(results):
    print(f'\n[Chunk {i+1}]')
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Text:   {doc.page_content[:300]}...')
    print('-' * 40)

## 8. Pipeline quality tests

Automated tests to verify the pipeline works correctly before moving to Notebook 02.

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('=' * 60)
print('PIPELINE QUALITY TESTS')
print('=' * 60)

tests_passed = 0
tests_failed = 0

def check(name, condition, detail=''):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f'  PASS - {name}')
    else:
        tests_failed += 1
        print(f'  FAIL - {name}')
    if detail:
        print(f'         {detail}')

# Test 1 - Pinecone index exists and has data
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
check('Pinecone index contains vectors',
    stats.total_vector_count > 0,
    f'Vectors in Pinecone: {stats.total_vector_count}')

# Test 2 - Similarity search returns results
results = vectorstore.similarity_search('high rise building fire', k=3)
check('Similarity search returns results',
    len(results) == 3,
    f'Got {len(results)} results')

# Test 3 - Retrieved chunks contain relevant content
combined = ' '.join([r.page_content.lower() for r in results])
keywords = ['fire', 'building', 'floor', 'high', 'smoke', 'brussels']
found = [k for k in keywords if k in combined]
check('Retrieved chunks contain relevant content',
    len(found) >= 3,
    f'Keywords found: {found}')

# Test 4 - Metadata present on retrieved chunks
check('Chunks have source metadata',
    all(r.metadata.get('source') for r in results),
    f'Source: {results[0].metadata.get("source", "missing")}')

# Test 5 - Dutch query retrieves relevant results
results_nl = vectorstore.similarity_search('fouten gemaakt tijdens het incident', k=3)
combined_nl = ' '.join([r.page_content.lower() for r in results_nl])
check('Dutch query retrieves relevant content',
    len(results_nl) > 0,
    f'Got {len(results_nl)} results for Dutch query')

# Test 6 - End-to-end RAG test
context = '\n\n'.join([r.page_content for r in results])
prompt = f'Answer based only on context.\nContext:\n{context}\nQuestion: What lessons were learned?\nAnswer:'
response = llm.invoke(prompt)
check('LLM generates meaningful answer from retrieved context',
    len(response.content) > 50,
    f'Answer length: {len(response.content)} characters')

print('\n' + '=' * 60)
print(f'RESULTS: {tests_passed} passed | {tests_failed} failed')
if tests_failed == 0:
    print('All tests passed - ready for Notebook 02!')
else:
    print('Fix failing tests before moving to Notebook 02.')
print('=' * 60)

---

## What we built in this notebook

| Step | What | Why |
|------|------|-----|
| 1 | Pinecone cache check | Never fetch same video twice from YouTube |
| 2 | YouTube transcript fetch | Only called once per video ever |
| 3 | Timestamp preservation | Agent can cite exact moments |
| 4 | RecursiveCharacterTextSplitter | Natural boundaries, context preserved |
| 5 | text-embedding-3-small | 1536 dims, cost-efficient, high quality |
| 6 | Pinecone cloud storage | Accessible from any machine, no local files |
| 7 | Similarity search verification | Confirms full pipeline end-to-end |

## Next: Notebook 02 - RAG Chain
Build the full question-answering chain on top of this knowledge base.